# GAVE2 CMRRWNet V7: Conditional A/V Paths

This clean Colab run warm-starts three full-resolution folds from the certified V6 checkpoints. V7 replaces independent artery/vein decisions with a conditional softmax, selects checkpoints with a connected-path metric, and cross-fits branch reconstruction before producing a certified submission. V6 files and submissions are read-only inputs.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

TEAM_ID = "\u68af\u5ea6\u4e0d\u4e0b\u964d\u961f"
DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_v7.zip"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
V6_RUN_DIR = DRIVE_BASE / "runs/gave2_cmrrwnet_v6_3fold"
RUN_DIR = DRIVE_BASE / "runs/gave2_cmrrwnet_v7_3fold"
FOLD_MANIFEST = V6_RUN_DIR / "fold_manifest.json"

V7_OOF_ROOT = RUN_DIR / "predictions/oof"
V7_VALIDATION_ROOT = RUN_DIR / "predictions/validation"
V7_PATH_OOF_ROOT = RUN_DIR / "predictions/path_oof"
V7_PATH_VALIDATION_ROOT = RUN_DIR / "predictions/path_validation"
V7_REPORT_ROOT = RUN_DIR / "path_reports"

V6_BASE_TEAM_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v6_3fold/base" / TEAM_ID
V6_REFINED_TEAM_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v6_3fold/refined" / TEAM_ID
CONTROL_OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v7_3fold/control_hybrid"
CONTROL_TEAM_ROOT = CONTROL_OUTPUT_ROOT / TEAM_ID
CONTROL_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v7_control_hybrid.zip"
CONTROL_REPORT = RUN_DIR / "control_hybrid_submission_report.json"
TASK12_OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v7_3fold/task12"
TASK12_TEAM_ROOT = TASK12_OUTPUT_ROOT / TEAM_ID
FINAL_OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v7_3fold/main"
FINAL_TEAM_ROOT = FINAL_OUTPUT_ROOT / TEAM_ID
FINAL_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v7_main.zip"
FINAL_REPORT = RUN_DIR / "main_submission_report.json"

N_FOLDS = 3
SEED = 77
TASK2_MAX_EPOCHS = 40
TASK1_MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7
NUM_REFINEMENTS = 2
AUTO_DISCONNECT = True

def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *[str(value) for value in arguments]]
    print("RUN:", " ".join(command), flush=True)
    return subprocess.run(command, cwd=WORK_ROOT, check=True, text=True)


## Mount Drive And Extract The Clean V7 Archive


In [ ]:
from google.colab import drive
from pathlib import PurePosixPath

drive.mount("/content/drive")
assert ARCHIVE_PATH.is_file(), f"Upload the new archive to {ARCHIVE_PATH}"

required_members = {
    "experiments/gave2_ensemble/train_v7.py",
    "experiments/gave2_ensemble/predict_v7.py",
    "experiments/gave2_ensemble/path_v7.py",
    "experiments/gave2_ensemble/submission_v7.py",
    "tests/gave2_ensemble/test_v7_core.py",
    "knowledge_base/sources/github/Peng2004_CMRRWNet/train/models.py",
}
anchor = "experiments/gave2_ensemble/train_v7.py"

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None, "Archive CRC validation failed"
    entries = archive.infolist()
    names = [entry.filename.replace("\\", "/").lstrip("/") for entry in entries]
    matches = [name for name in names if name.endswith(anchor)]
    assert len(matches) == 1, f"Expected one V7 anchor, found {matches}"
    prefix = matches[0][:-len(anchor)]
    relative_names = {name[len(prefix):] for name in names if name.startswith(prefix)}
    missing = sorted(required_members - relative_names)
    assert not missing, f"V7 archive is missing: {missing}"
    assert any(name.startswith("GAVE2_preliminary/") for name in relative_names), "V7 archive has no dataset"

    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    WORK_ROOT.mkdir(parents=True)
    allowed_roots = {"GAVE2_preliminary", "experiments", "tests", "knowledge_base"}
    root = WORK_ROOT.resolve()
    for entry, normalized in zip(entries, names):
        if not normalized.startswith(prefix):
            continue
        relative = normalized[len(prefix):].lstrip("/")
        if not relative:
            continue
        parts = PurePosixPath(relative).parts
        if not parts or parts[0] not in allowed_roots or ".." in parts:
            continue
        destination = WORK_ROOT.joinpath(*parts)
        resolved = destination.resolve()
        if root != resolved and root not in resolved.parents:
            raise RuntimeError(f"Unsafe archive member: {entry.filename}")
        if entry.is_dir() or relative.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

assert DATA_ROOT.is_dir()
assert FOLD_MANIFEST.is_file(), f"Missing V6 fold manifest on Drive: {FOLD_MANIFEST}"
for task in ("task1", "task2"):
    for fold in range(N_FOLDS):
        fold_root = V6_RUN_DIR / "cmrrwnet_v6" / task / f"fold_{fold}"
        assert (fold_root / "best.pt").is_file(), f"Missing V6 warm start: {fold_root / 'best.pt'}"
        assert (fold_root / "best.certified.json").is_file()
for source in (V6_BASE_TEAM_ROOT, V6_REFINED_TEAM_ROOT):
    for task_name in ("Task1", "Task2", "Task3"):
        assert (source / task_name).is_dir(), f"Missing V6 submission source: {source / task_name}"

print({"archive": str(ARCHIVE_PATH), "members": len(entries), "data": str(DATA_ROOT), "v6": str(V6_RUN_DIR)})


## Install Dependencies And Enforce Reproducible BF16 CUDA


In [ ]:
requirements = WORK_ROOT / "experiments/gave2_ensemble/requirements-gave2-main.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "pytest"], check=True)

import torch

assert torch.cuda.is_available(), "V7 full-resolution training requires a CUDA GPU"
assert torch.cuda.is_bf16_supported(), "V7 requires a CUDA GPU with BF16 support"
gpu = torch.cuda.get_device_properties(0)

test_files = [
    "tests/gave2_ensemble/test_v7_core.py",
    "tests/gave2_ensemble/test_data_v6.py",
    "tests/gave2_ensemble/test_prediction_v6.py",
]
subprocess.run([sys.executable, "-m", "pytest", *test_files, "-q"], cwd=WORK_ROOT, check=True)

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.cmrrwnet_v7 import create_cmrrwnet_v7
from experiments.gave2_ensemble.losses_v7 import ConditionalPathLossV7, conditional_probabilities_from_logits

for task, channels in (("task1", 4), ("task2", 6)):
    model = create_cmrrwnet_v7(task, base_channels=4, num_refinements=1, activation_checkpointing=False).cuda()
    image = torch.zeros(1, channels, 32, 32, device="cuda")
    target = torch.zeros(1, 3, 32, 32, device="cuda")
    mask = torch.ones(1, 1, 32, 32, device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        predictions = model(image)
        loss = ConditionalPathLossV7([2.0, 2.0, 2.0])(predictions, target, mask)
    loss.backward()
    probability = conditional_probabilities_from_logits(predictions)
    assert not bool(((probability[:, 0] >= 0.5) & (probability[:, 2] >= 0.5)).any())
    del model, image, target, mask, predictions, probability, loss
    torch.cuda.empty_cache()

print({"torch": torch.__version__, "gpu": gpu.name, "vram_gib": round(gpu.total_memory / 1024**3, 2), "bf16": True})


## Certify The Base-Task1/2 Plus Refined-Task3 Control


In [ ]:
from experiments.gave2_ensemble.submission_v7 import assemble_hybrid, certify_v7
from experiments.gave2_ensemble.submission_v6 import readback_zip

RUN_DIR.mkdir(parents=True, exist_ok=True)
if not CONTROL_ZIP.exists():
    if CONTROL_TEAM_ROOT.exists():
        shutil.rmtree(CONTROL_TEAM_ROOT)
    assemble_hybrid(V6_BASE_TEAM_ROOT, V6_REFINED_TEAM_ROOT, CONTROL_TEAM_ROOT)
    certify_v7(CONTROL_TEAM_ROOT, DATA_ROOT, CONTROL_ZIP, CONTROL_REPORT)
control_readback = readback_zip(CONTROL_ZIP, TEAM_ID)
assert control_readback["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print({"control_zip": str(CONTROL_ZIP), "expected_preliminary_score": 6.02092, "sha256": control_readback["sha256"]})


## Select The Largest Safe BF16 Batch On The Current GPU


In [ ]:
from experiments.gave2_ensemble.memory_test_v7 import parse_args as parse_memory_args, run_memory_test

BASE_CHANNELS = {}
MEMORY_PROFILES = {}
for task in ("task2", "task1"):
    v6_config = json.loads((V6_RUN_DIR / "cmrrwnet_v6" / task / "fold_0/config.json").read_text())
    BASE_CHANNELS[task] = int(v6_config["base_channels"])
    profile = run_memory_test(parse_memory_args([
        "--data-root", str(DATA_ROOT), "--fold-manifest", str(FOLD_MANIFEST),
        "--task", task, "--base-channels", str(BASE_CHANNELS[task]),
        "--num-refinements", str(NUM_REFINEMENTS), "--steps", "2",
    ]))
    MEMORY_PROFILES[task] = profile
(RUN_DIR / "memory_profiles.json").write_text(json.dumps(MEMORY_PROFILES, indent=2))
print(json.dumps(MEMORY_PROFILES, indent=2))


## Warm-Start And Fine-Tune All Three V7 Folds


In [ ]:
def train_all_folds(task, epochs):
    profile = MEMORY_PROFILES[task]
    batch_size = int(profile["batch_size"])
    grad_accum = 2 if batch_size == 2 else 1
    for fold in range(N_FOLDS):
        fold_dir = RUN_DIR / "cmrrwnet_v7" / task / f"fold_{fold}"
        arguments = [
            "--data-root", DATA_ROOT, "--run-dir", RUN_DIR,
            "--warm-start-run-dir", V6_RUN_DIR, "--fold-manifest", FOLD_MANIFEST,
            "--task", task, "--fold", fold, "--base-channels", BASE_CHANNELS[task],
            "--num-refinements", NUM_REFINEMENTS, "--batch-size", batch_size,
            "--grad-accum", grad_accum, "--workers", 2, "--epochs", epochs,
            "--amp", "bf16", "--lr", 8e-5, "--min-lr", 5e-6,
            "--weight-decay", 1e-4, "--grad-clip", 1.0,
            "--early-stopping-patience", EARLY_STOPPING_PATIENCE,
            "--early-stopping-min-delta", 2e-4, "--seed", SEED,
        ]
        if not profile["activation_checkpointing"]:
            arguments.append("--no-activation-checkpointing")
        if (fold_dir / "last.pt").is_file():
            arguments.append("--resume")
        run_module("experiments.gave2_ensemble.train_v7", *arguments)

train_all_folds("task2", TASK2_MAX_EPOCHS)
train_all_folds("task1", TASK1_MAX_EPOCHS)
print("All V7 folds trained, resumed exactly, or stopped after seven stale epochs.")


## Generate Exclusive V7 OOF Probabilities


In [ ]:
for task in ("task2", "task1"):
    run_module(
        "experiments.gave2_ensemble.predict_v7",
        "--data-root", DATA_ROOT, "--run-dir", RUN_DIR, "--fold-manifest", FOLD_MANIFEST,
        "--store-root", V7_OOF_ROOT / task, "--task", task, "--mode", "oof",
    )
print("V7 OOF stores complete.")


## Cross-Fit Connected-Path Reconstruction


In [ ]:
PATH_REPORTS = {}
for task in ("task2", "task1"):
    report = V7_REPORT_ROOT / f"{task}_path_gate.json"
    run_module(
        "experiments.gave2_ensemble.path_v7", "fit",
        "--data-root", DATA_ROOT, "--fold-manifest", FOLD_MANIFEST,
        "--source-store-root", V7_OOF_ROOT / task,
        "--crossfit-output-store-root", V7_PATH_OOF_ROOT / task,
        "--task", task, "--output", report,
    )
    PATH_REPORTS[task] = report
    payload = json.loads(report.read_text())
    print(task, json.dumps({"gate": payload["gate"], "raw": payload["raw_metrics"], "crossfit": payload["crossfit_metrics"]}, indent=2))


## Validation Inference And Gate-Controlled Promotion


In [ ]:
for task in ("task2", "task1"):
    run_module(
        "experiments.gave2_ensemble.predict_v7",
        "--data-root", DATA_ROOT, "--run-dir", RUN_DIR, "--fold-manifest", FOLD_MANIFEST,
        "--store-root", V7_VALIDATION_ROOT / task, "--task", task, "--mode", "validation",
    )
    run_module(
        "experiments.gave2_ensemble.path_v7", "apply",
        "--data-root", DATA_ROOT, "--source-store-root", V7_VALIDATION_ROOT / task,
        "--output-store-root", V7_PATH_VALIDATION_ROOT / task,
        "--report", PATH_REPORTS[task], "--task", task, "--source-split", "validation",
    )
    run_module(
        "experiments.gave2_ensemble.path_v7", "promote",
        "--data-root", DATA_ROOT, "--source-store-root", V7_PATH_VALIDATION_ROOT / task,
        "--output-root", TASK12_OUTPUT_ROOT, "--team-id", TEAM_ID, "--task", task,
    )
print("Task 1 and Task 2 V7 outputs complete.")


## Assemble V7 With The Proven Refined Task 3 And Certify


In [ ]:
from experiments.gave2_ensemble.submission_v7 import assemble_v7

if not FINAL_ZIP.exists():
    if FINAL_TEAM_ROOT.exists():
        shutil.rmtree(FINAL_TEAM_ROOT)
    assemble_v7(TASK12_TEAM_ROOT, V6_REFINED_TEAM_ROOT, FINAL_TEAM_ROOT)
    certify_v7(FINAL_TEAM_ROOT, DATA_ROOT, FINAL_ZIP, FINAL_REPORT)
final_readback = readback_zip(FINAL_ZIP, TEAM_ID)
assert final_readback["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print({"ready_to_submit": str(FINAL_ZIP), "sha256": final_readback["sha256"], "counts": final_readback["counts"]})


## Release The Runtime After Both ZIPs Pass Readback


In [ ]:
assert readback_zip(CONTROL_ZIP, TEAM_ID)["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
assert readback_zip(FINAL_ZIP, TEAM_ID)["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print({"control": str(CONTROL_ZIP), "main": str(FINAL_ZIP), "path_gates": {task: json.loads(path.read_text())["gate"] for task, path in PATH_REPORTS.items()}})
if AUTO_DISCONNECT:
    print("Certified V7 artifacts are on Drive. Disconnecting in 10 seconds.", flush=True)
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()
